## CIFAR-10 Classification with a Convolutional Neural Network

CIFAR-10 contains 60 000 colour images (32×32 px, 3 channels) spread evenly across 10 classes:
plane, car, bird, cat, deer, dog, frog, horse, ship, truck.

This notebook follows the same structure as the FashionMNIST CNN notebook:
1. **Loading & visualising** the raw data and class distribution
2. **Augmentation preview** – what the training transforms look like on a single image
3. **Baseline MLP** – a fully-connected network on flattened pixels, used as a reference
4. **CNN** – two convolutional blocks followed by a small classifier head
5. **Feature-map inspection** – what the last conv block detects
6. **Side-by-side predictions** – MLP vs CNN on the same test images
7. **Evaluation** – accuracy, classification report, confusion matrix for both models


In [1]:
# Import necessary libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import numpy as np
import matplotlib.pyplot as plt

# Set up device (is available use GPU to speed up computations)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


Load CIFAR-10 with a minimal `ToTensor()` transform — just enough to inspect the raw images and the class distribution before deciding on normalisation.


In [ ]:
import torchvision
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

# Function to show a grid of images (RGB, no unnormalisation needed here)
def imshow(img):
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()

# Load CIFAR-10 with a minimal transform for visualisation only
transform = transforms.Compose([transforms.ToTensor()])

train_dataset = datasets.CIFAR10(root='./data', train=True,  download=True,  transform=transform)
test_dataset  = datasets.CIFAR10(root='./data', train=False, download=True,  transform=transform)

# Sample batch
dataiter = iter(DataLoader(train_dataset, batch_size=4, shuffle=True))
images, labels = next(dataiter)

print('Sample images:')
imshow(torchvision.utils.make_grid(images))

classes = ('plane', 'car', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck')
print('Labels:', ' '.join(f'{classes[labels[j]]:10s}' for j in range(4)))

# Class distribution
class_counts = {c: 0 for c in classes}
for _, label in train_dataset:
    class_counts[classes[label]] += 1

plt.figure(figsize=(10, 4))
plt.bar(class_counts.keys(), class_counts.values(), color='skyblue')
plt.xlabel('Class')
plt.ylabel('Count')
plt.title('Class distribution – CIFAR-10 training set')
plt.show()


Apply the augmentation pipeline to the same image 10 times to get an intuition for how much each transformation shifts the appearance. CIFAR-10 images are small (32×32) so the rotation and shear are intentionally mild.


In [ ]:
import matplotlib.pyplot as plt
import torchvision.transforms as transforms

def visualize_single_image_augmentations(image, transform, repetitions=10):
    fig, axes = plt.subplots(1, repetitions, figsize=(20, 2))
    for i in range(repetitions):
        img = transform(image).numpy()
        # Unnormalise from [-1, 1] to [0, 1] for display
        img = (img * 0.5 + 0.5).clip(0, 1)
        axes[i].imshow(np.transpose(img, (1, 2, 0)))
        axes[i].axis('off')
    plt.suptitle('Same image, 10 different augmentations', y=1.02)
    plt.tight_layout()
    plt.show()

# Load a single raw image (no transform) from the training set
original_image, _ = datasets.CIFAR10(root='./data', train=True, download=False,
                                      transform=None)[0]

augmentation_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1), shear=10),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

visualize_single_image_augmentations(original_image, augmentation_transform)


Define the normalisation and augmentation transforms, then carve out a 20 % validation split.

We load the raw training data **twice** — once with the augmented transform (for training) and once with the plain transform (for validation). Both are sliced with the **same random indices**, so train and validation never overlap. This avoids the aliasing problem that arises when using `random_split` with a shared dataset object.


In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
import torch

# ── Transforms ──────────────────────────────────────────────────────────────
# CIFAR-10 is RGB, so we normalise all three channels
train_transform_augment = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.RandomAffine(0, shear=10, scale=(0.8, 1.2)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# ── Load the dataset twice — same images, different transforms ──────────────
# download=False because the files are already on disk from the first cell
train_dataset_aug   = datasets.CIFAR10(root='./data', train=True,
                                        download=False,
                                        transform=train_transform_augment)
train_dataset_plain = datasets.CIFAR10(root='./data', train=True,
                                        download=False,
                                        transform=test_transform)

# ── Split with the same indices ─────────────────────────────────────────────
validation_split = 0.2
n               = len(train_dataset_aug)
train_size      = int((1 - validation_split) * n)

# Fix the seed so the split is reproducible across runs
indices = torch.randperm(n, generator=torch.Generator().manual_seed(42))

train_data      = Subset(train_dataset_aug,   indices[:train_size].tolist())
validation_data = Subset(train_dataset_plain, indices[train_size:].tolist())

print(f"Training Data:    {len(train_data)} samples")
print(f"Validation Data:  {len(validation_data)} samples")
print(f"Test Data:        {len(test_dataset)} samples")


**`EarlyStopping`** – halts training if the validation loss fails to improve by at least `min_delta` for `patience` consecutive epochs. Saves a checkpoint at every new best.

**`train_model`** – a single training loop that works for both architectures (returns a single logit tensor).


In [ ]:
# EARLY STOPPING AND TRAINING FUNCTION

class EarlyStopping:
    def __init__(self, patience=25, min_delta=0.0001, path='checkpoint.pt'):
        self.patience   = patience
        self.min_delta  = min_delta
        self.counter    = 0
        self.best_loss  = None
        self.early_stop = False
        self.path       = path

    def __call__(self, val_loss, model):
        if self.best_loss is None:
            self.best_loss = val_loss
            self.save_checkpoint(val_loss, model, first=True)
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.save_checkpoint(val_loss, model)
            self.best_loss = val_loss
            self.counter   = 0

    def save_checkpoint(self, val_loss, model, first=False):
        torch.save(model.state_dict(), self.path)
        if first:
            print(f'Saving initial model checkpoint (loss: {val_loss:.6f}) ...')
        else:
            print(f'Validation loss decreased ({self.best_loss:.6f} --> {val_loss:.6f}).  Saving model ...')


def train_model(epochs, model, optimizer, criterion, train_loader, val_loader, early_stopper):
    train_losses, val_losses = [], []
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for data, target in train_loader:
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            output = model(data)
            loss   = criterion(output, target)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_losses.append(train_loss / len(train_loader))

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for data, target in val_loader:
                data, target = data.to(device), target.to(device)
                output = model(data)
                loss   = criterion(output, target)
                val_loss += loss.item()
        val_losses.append(val_loss / len(val_loader))
        print(f'Epoch: {epoch+1}, Training Loss: {train_loss/len(train_loader):.4f}, '
              f'Validation Loss: {val_loss/len(val_loader):.4f}')

        early_stopper(val_loss / len(val_loader), model)
        if early_stopper.early_stop:
            print("Early stopping triggered.")
            break

    # weights_only=True suppresses a DeprecationWarning in recent PyTorch versions
    model.load_state_dict(torch.load('checkpoint.pt', weights_only=True))
    return train_losses, val_losses


## Baseline: MLP

Before training the CNN, we fit a simple fully-connected network on the same data.
CIFAR-10 images have **32×32×3 = 3 072** input pixels, which makes the MLP significantly
wider than it was for FashionMNIST (784 pixels). The spatial structure of the image
is completely discarded — every pixel is treated as an independent feature.
This gives us a concrete benchmark to measure how much the convolutional layers actually help.


In [ ]:
import torch.nn as nn
from torchsummary import summary

class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.flatten = nn.Flatten()
        self.network = nn.Sequential(
            nn.Linear(32 * 32 * 3, 512),  # 3 072 input features (RGB)
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 10),            # 10 output classes
        )

    def forward(self, x):
        x = self.flatten(x)
        return self.network(x)

mlp_model = MLP().to(device)
summary(mlp_model, input_size=(3, 32, 32))


Train the MLP with early stopping and plot the loss curves.
We use a lower learning rate than the CNN (1e-4 vs 1e-3) — fully-connected networks
on image data tend to be less stable with higher rates.


In [ ]:
batch_size = 512
epochs     = 10
patience   = 5
lr_mlp     = 0.0001
reg        = 0.0001

train_loader      = DataLoader(train_data,      batch_size=batch_size, shuffle=True)
validation_loader = DataLoader(validation_data, batch_size=batch_size, shuffle=False)
test_loader       = DataLoader(test_dataset,    batch_size=batch_size, shuffle=False)

criterion     = nn.CrossEntropyLoss()
mlp_optimizer = optim.Adam(mlp_model.parameters(), lr=lr_mlp, weight_decay=reg)
early_stopper = EarlyStopping(patience=patience)

train_losses_mlp, val_losses_mlp = train_model(
    epochs, mlp_model, mlp_optimizer, criterion,
    train_loader, validation_loader, early_stopper
)

plt.plot(train_losses_mlp, label='Training Loss')
plt.plot(val_losses_mlp,   label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('MLP – Loss curves')
plt.legend()
plt.show()


The architecture mirrors the FashionMNIST CNN, adapted for CIFAR-10's **32×32 RGB** inputs:

- The first conv layer takes 3 input channels instead of 1
- After two MaxPool layers the spatial size is 8×8 (vs 7×7 for FashionMNIST)
- The flattened vector is therefore 64×8×8 = 4 096 features

`forward()` returns **only logits** — no softmax, because `CrossEntropyLoss` fuses log-softmax and NLL internally. Feature maps are extracted separately during visualisation by calling `model.features()` directly.


In [ ]:
import torch.nn as nn
from torchsummary import summary

class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.features = nn.Sequential(
            # Input: (3, 32, 32) — RGB image
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            # Output: (32, 32, 32)  |  32 filters of shape (3, 3, 3)
            nn.ReLU(),

            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            # Output: (32, 32, 32)  |  32 filters of shape (32, 3, 3)
            nn.ReLU(),

            nn.MaxPool2d(2, 2),
            # Output: (32, 16, 16) — halves spatial dimensions

            nn.Dropout(0.25),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            # Output: (64, 16, 16)  |  64 filters of shape (32, 3, 3)
            nn.ReLU(),

            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            # Output: (64, 16, 16)  |  64 filters of shape (64, 3, 3)
            nn.ReLU(),

            nn.MaxPool2d(2, 2),
            # Output: (64, 8, 8)

            nn.Dropout(0.25)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            # Output: 64 × 8 × 8 = 4096

            nn.Linear(64 * 8 * 8, 64),
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.Linear(64, 10)
            # Raw logits — no softmax. CrossEntropyLoss = LogSoftmax + NLLLoss internally
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x   # logits only

cnn_model = CNN().to(device)
summary(cnn_model, input_size=(3, 32, 32))


Train the CNN with early stopping and plot the loss curves.


In [ ]:
lr  = 0.001
reg = 0.0001

criterion     = nn.CrossEntropyLoss()
cnn_optimizer = optim.Adam(cnn_model.parameters(), lr=lr, weight_decay=reg)
early_stopper = EarlyStopping(patience=patience)

train_losses_cnn, val_losses_cnn = train_model(
    epochs, cnn_model, cnn_optimizer, criterion,
    train_loader, validation_loader, early_stopper
)

plt.plot(train_losses_cnn, label='Training Loss')
plt.plot(val_losses_cnn,   label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('CNN – Loss curves')
plt.legend()
plt.show()


Visualise what the last convolutional block responds to on CIFAR-10 images.
Each row shows one input image alongside the first `num_maps` feature maps produced after the second MaxPool.

Because `forward()` returns only logits, we extract feature maps by calling `model.features(images)` directly — no changes to the model needed.


In [ ]:
import matplotlib.pyplot as plt

def visualize_features_and_inputs(model, data_loader, num_images=5, num_maps=5):
    model.eval()
    fig, axes = plt.subplots(num_images, num_maps + 1, figsize=(15, 2.5 * num_images))

    # Fetch one batch
    images, _ = next(iter(data_loader))
    images = images.to(device)

    # Extract feature maps directly from the convolutional block
    with torch.no_grad():
        feature_maps = model.features(images).cpu()

    for idx in range(num_images):
        # Display the input image — unnormalise from [-1, 1] to [0, 1]
        img = images[idx].cpu().numpy()
        img = (img * 0.5 + 0.5).clip(0, 1)
        axes[idx, 0].imshow(np.transpose(img, (1, 2, 0)))
        axes[idx, 0].set_title('Input' if idx == 0 else '')
        axes[idx, 0].axis('off')

        for i in range(num_maps):
            if i < feature_maps.size(1):
                axes[idx, i + 1].imshow(feature_maps[idx, i], cmap='hot')
                axes[idx, i + 1].set_title(f'Map {i+1}' if idx == 0 else '')
                axes[idx, i + 1].axis('off')
            else:
                axes[idx, i + 1].axis('off')

    plt.tight_layout()
    plt.show()

visualize_features_and_inputs(cnn_model, test_loader, num_images=5, num_maps=7)


Run both models on the same batch of test images and display the predictions side by side.
Cases where the MLP is wrong but the CNN is right reveal what convolutional feature
extraction adds over a flat pixel representation.


In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np

mlp_model.eval()
cnn_model.eval()

N_samples = 8

# Fresh loader so we get a random batch each run
pred_loader = DataLoader(test_dataset, batch_size=N_samples, shuffle=True)
images, labels = next(iter(pred_loader))
images, labels = images.to(device), labels.to(device)

with torch.no_grad():
    outputs_mlp = mlp_model(images)
    outputs_cnn = cnn_model(images)

_, predicted_mlp = torch.max(outputs_mlp, 1)
_, predicted_cnn = torch.max(outputs_cnn, 1)

images_cpu = images.cpu().numpy()
labels_cpu = labels.cpu().numpy()
pred_mlp   = predicted_mlp.cpu().numpy()
pred_cnn   = predicted_cnn.cpu().numpy()

fig, axes = plt.subplots(1, N_samples, figsize=(15, 3))
for i in range(N_samples):
    # Unnormalise from [-1, 1] to [0, 1] for display
    img = (images_cpu[i] * 0.5 + 0.5).clip(0, 1)
    axes[i].imshow(np.transpose(img, (1, 2, 0)))
    axes[i].set_title(
        f"True: {classes[labels_cpu[i]]}\n"
        f"MLP:  {classes[pred_mlp[i]]}\n"
        f"CNN:  {classes[pred_cnn[i]]}",
        fontsize=7
    )
    axes[i].axis('off')

plt.tight_layout()
plt.show()


Full evaluation on the validation set: accuracy, per-class precision/recall/F1, and the confusion matrix.


In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

def evaluate_model(data_loader, model, label='Model'):
    model.eval()
    y_true, y_preds = [], []
    with torch.no_grad():
        for images, labels in data_loader:
            images, labels = images.to(device), labels.to(device)
            outputs      = model(images)
            _, predicted = torch.max(outputs, 1)
            y_preds.extend(predicted.cpu().numpy())
            y_true.extend(labels.cpu().numpy())

    accuracy = accuracy_score(y_true, y_preds)
    print(f"Accuracy: {accuracy:.4f}")
    print("Classification Report:\n",
          classification_report(y_true, y_preds, target_names=classes, digits=4))
    print("Confusion Matrix:\n", confusion_matrix(y_true, y_preds))
    return accuracy

print("\n---------------MLP MODEL---------------\n")
mlp_accuracy = evaluate_model(validation_loader, mlp_model, label='MLP')

print("\n---------------CNN MODEL---------------\n")
cnn_accuracy = evaluate_model(validation_loader, cnn_model, label='CNN')

print(f"\nMLP accuracy: {mlp_accuracy:.4f}")
print(f"CNN accuracy: {cnn_accuracy:.4f}")
print(f"CNN improvement over MLP: {(cnn_accuracy - mlp_accuracy)*100:+.2f} pp")
